### Deformation Stress and Strain

####  *AUTHOR:* Benjamin Scullett
####  *CONTACT:* benjamin.scullett@sydney.edu.au
####  *DATE last modified:* 02/10/2025

Ehsan knows this better than I.

We begin by importing the required libraries:

In [1]:
# Import libraries
from ipywidgets import interact
import os

import cartopy.crs as ccrs
import cmcrameri.cm as ccm
import gplately
from gplately import PlateReconstruction, PlotTopologies
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
import matplotlib.pyplot as plt
import pandas as pd

# Note: Ensure the 'lib' folder is located in the same folder as this notebook
from lib.reconstruction_coregistration import *
from lib.deformation_parallel import *
from lib.feature_extraction import *
from lib.plot import *

# Load configuration parameters (e.g., paths, model names)
from parameters import parameters

2025-10-08 15:08:19 - gplately - ERROR - Failed to import PyGMT. PyGMT requires Python>=3.11.
2025-10-08 15:08:20 - gplately - ERROR - Failed to import PyGMT. PyGMT requires Python>=3.11.


### Setup

As defined in `parameters.py`, the cell below configures the analysis parameters and specifies the paths to input/output files and directories. You can also specify the number of cores to use by setting an appropriate value for the `n_jobs` variable at the end of the cell.

**Note:** You can modify the analysis settings directly in `parameters.py`, located in the same folder as this notebook. The file is structured as a dictionary; look for keys such as `timespan` and `grid_resolution` to adjust their values as needed.

In [2]:
# Set up temporal analysis parameters
temporal_resolution = parameters["temporal_resolution"]
time_min = parameters["timespan"]["min"] # Youngest time to analyse
time_max = parameters["timespan"]["max"] # Oldest time to analyse
# Create an array of time steps for analysis (e.g., 0, 1, 2, ... Ma)
time_steps = range(time_min, time_max + temporal_resolution, temporal_resolution)

buffer_distance = parameters["buffer_distance"]
num_random = parameters["num_random"] # Number of random points to be generated at each time step
grid_resolution = parameters["grid_resolution"]

# Directory paths for inputs and outputs
plate_model_dir = parameters["plate_model_dir"]
inputs_dir = parameters["inputs_dir"]
outputs_dir = parameters["outputs_dir"]
buffer_zones_dir = parameters["buffer_zones_dir"]

# Data filenames
subduction_data_filename = parameters["subduction_data_filename"]
deposit_coords_filename = parameters["deposit_coords_filename"]
deposit_coords_recon_filename = parameters["deposit_coords_recon_filename"]
deposit_coords_recon_all_filename = parameters["deposit_coords_recon_all_filename"]
unlabelled_coords_filename = parameters["unlabelled_coords_filename"]
backarc_coords_filename = parameters["backarc_coords_filename"]
deposit_data_filename = parameters["deposit_data_filename"]
unlabelled_data_filename = parameters["unlabelled_data_filename"]
backarc_data_filename = parameters["backarc_data_filename"]

buffer_zones_dir = os.path.join(outputs_dir, buffer_zones_dir)

# Construct full file paths
subduction_data_filename = os.path.join(outputs_dir, subduction_data_filename)
deposit_coords_filename = os.path.join(inputs_dir, deposit_coords_filename)
deposit_coords_recon_filename = os.path.join(outputs_dir, deposit_coords_recon_filename)
deposit_coords_recon_all_filename = os.path.join(outputs_dir, deposit_coords_recon_all_filename)
unlabelled_coords_filename = os.path.join(outputs_dir, unlabelled_coords_filename)
backarc_coords_filename = os.path.join(outputs_dir, backarc_coords_filename)
deposit_data_filename = os.path.join(outputs_dir, deposit_data_filename)
unlabelled_data_filename = os.path.join(outputs_dir, unlabelled_data_filename)
backarc_data_filename = os.path.join(outputs_dir, backarc_data_filename)

# Paths to different types of grids
agegrid_dir = os.path.join(inputs_dir, "SeafloorAge")
crusthick_dir = os.path.join(inputs_dir, "CrustalThickness")

# Number of cores to be used for running this notebook
n_jobs = 5

The cell below loads the necessary files to create the `PlateReconstruction` and `PlateTopologies` objects, which will later be used for reconstruction and generating visualisations. Moreover, the feature values calculated using the `01_feature_extraction` notebook will also be loaded in this cell.

In [3]:
# STELLAR4A plate motion model
rotation_model = plate_model_dir+"/CombinedRotations.rot"

topology_features = [
    plate_model_dir+"/Deforming_Networks_Active.gpml",
    plate_model_dir+"/Deforming_Networks_Inactive.gpml",
    plate_model_dir+"/Feature_Geometries.gpml",
    # plate_model_dir+"/Flat_Slabs.gpml", # Including this file may introduce artifacts, so it is recommended to exclude it when creating the PlateReconstruction object.
    plate_model_dir+"/Plate_Boundaries.gpml",
]

static_polygons = plate_model_dir+"/Global_EarthByte_GPlates_PresentDay_StaticPlatePolygons.gpml"
coastlines = plate_model_dir+"/Global_coastlines_low_res.gpml"
continents = plate_model_dir+"/Global_EarthByte_GPlates_PresentDay_ContinentsAndArcs.gpml"
COBs = plate_model_dir+"/Global_EarthByte_GeeK07_COBLineSegments_2019_v1.gpml"

plate_reconstruction = PlateReconstruction(
    rotation_model=rotation_model,
    topology_features=topology_features,
    static_polygons=static_polygons,
)

gplot = PlotTopologies(
    plate_reconstruction=plate_reconstruction,
    coastlines=coastlines,
    continents=continents,
    COBs=COBs,
)

subduction_data = pd.read_csv(subduction_data_filename) # Load feature values
# deposit_coords = pd.read_csv(deposit_coords_filename) # Load deposit coordinates and weights

# Define map projection (Mollweide provides a good global view)
projection = ccrs.Mollweide(central_longitude=60)

The cell below is interactive, allowing users to select a geological time and visualize buffer zones, seafloor age, and plate boundaries — including mid-ocean ridges and transform faults.

### Buffer Zones

This cell creates one-sided buffer zones around trench lines within subduction zones at each time step, covering arc–backarc environments. The buffer width can be customised using the `buffer_distance` argument in the function (default: 6°). You can also adjust this value in the `parameters.py` file. In the following cells, random (unlabeled) points and target points are generated within these buffer zones.

In [4]:
# Create buffer zones if the relevant directory does not exist
if not os.path.isdir(buffer_zones_dir):
    run_create_buffer_zones(
        times=time_steps,
        rotation_model=rotation_model,
        topology_features=topology_features,
        static_polygons=static_polygons,
        output_dir=buffer_zones_dir,
        buffer_distance=buffer_distance, # Width of the buffer zones
        n_jobs=n_jobs,
        verbose=True,
        return_output=False,
    )

The cell below is interactive, allowing users to select a geological time and plot reconstructed mineral occurrences. Seafloor age and plate boundaries — including mid-ocean ridges and transform faults — are also plotted.

The cell below is interactive, allowing users to select a geological time and plot unlabelled samples. Seafloor age and plate boundaries — including mid-ocean ridges and transform faults — are also plotted.

### Back-Arc Points

The cell below generates a grid of points at a user-defined resolution within previously generated buffer zones. You can adjust the resolution either by modifying the `resolution` argument in the function or by updating the value in the `parameters.py` file. The machine learning model will then be used to predict mineralisation at these locations.

In [5]:
# Generate grid points if the relevant file does not exist
if os.path.isfile(backarc_coords_filename):
    backarc_coords = pd.read_csv(backarc_coords_filename)
    backarc_coords = backarc_coords.dropna(subset=["present_lon", "present_lat"])
else:
    backarc_coords = generate_grid_points(
        times=time_steps,
        resolution=grid_resolution,
        polygons_dir=buffer_zones_dir,
        rotation_model=rotation_model,
        topology_features=topology_features,
        static_polygons=static_polygons,
        output_filename=backarc_coords_filename,
        n_jobs=n_jobs,
        verbose=True,
    )
    
    backarc_coords = backarc_coords.dropna(subset=["present_lon", "present_lat"])

The cell below is interactive, allowing users to select a geological time and plot grid points generated within the previously generated buffer zones. Seafloor age and plate boundaries — including mid-ocean ridges and transform faults — are also plotted.

### Coregistration

This code block performs co-registration by finding the closest trench point to each point in the previously created point sets — including reconstructed mineral occurrences, unlabelled samples, and grid points within the buffer. It then assigns the feature values of the nearest trench point to each of these points, along with the distance between them. The `run_coregister_crustal_thickness` function is specifically designed to coregister points with crustal thickness grids.

In [6]:
# if not os.path.isfile(backarc_data_filename):
#     backarc_data = run_coregister_point_data(
#         point_data=backarc_coords,
#         subduction_data=subduction_data,
#         n_jobs=n_jobs,
#         verbose=True,
#     )
    
#     backarc_data = run_coregister_crustal_thickness(
#         point_data=backarc_data,
#         input_dir=crusthick_dir,
#         output_filename=backarc_data_filename,
#         n_jobs=n_jobs,
#         verbose=True,
#     )

### Add strain rate and cumulative strain

In [7]:
deformation_data_filename = "outputs/deformation_data.csv"
if not os.path.isfile(deformation_data_filename):
    deformation_data = run_coregister_point_data(
        point_data=backarc_coords,
        subduction_data=subduction_data,
        n_jobs=n_jobs,
        verbose=True,
    )

    deformation_data = extract_strain_and_rate(
        deformation_data,
        topological_model=plate_reconstruction,
        max_time=time_max,
        n_jobs=n_jobs,
    )

    # Removing unwanted columns
    deformation_data = deformation_data.drop(columns=deformation_data.columns[[2, 3]])
    deformation_data = deformation_data.drop(columns=deformation_data.columns[range(3,25)])

    # Exporting data to csv
    deformation_data.to_csv(deformation_data_filename, index=False)

[Parallel(n_jobs=5)]: Using backend LokyBackend with 5 concurrent workers.
[Parallel(n_jobs=5)]: Done   2 out of   5 | elapsed:    9.6s remaining:   14.5s
[Parallel(n_jobs=5)]: Done   5 out of   5 | elapsed:   10.1s finished
